#Chile - Universidad Adolfo Ibáñez (UAI)
##Curso NLP
### Question Answering

# QA (Pipeline)

In [1]:
# Instalar Librerías
!pip install -U datasets huggingface_hub fsspec --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 563.3/563.3 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 21.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


In [2]:
#Importar Librerías
from transformers import pipeline

In [3]:
#Cargar el pipeline de QA (Español)
qa_pipeline = pipeline("question-answering",
                       model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Some weights of the model checkpoint at mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Fetching 0 files: 0it [00:00, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 0 files: 0it [00:00, ?it/s]

Device set to use cuda:0


In [4]:
#Contexto (Español)
contexto = """
La Organización Mundial de la Salud (OMS) fue fundada en 1948 y tiene su sede en Ginebra, Suiza.
Su objetivo principal es garantizar la salud global, coordinando respuestas ante pandemias y promoviendo la investigación médica.
"""

In [5]:
#Pregunta
pregunta = "¿Cuándo fue fundada la OMS?"
respuesta = qa_pipeline(question=pregunta, context=contexto)
print(f"Pregunta: {pregunta}")
print(f"Respuesta: {respuesta['answer']}\n")

Pregunta: ¿Cuándo fue fundada la OMS?
Respuesta: 1948



In [6]:
#Listado de Preguntas
preguntas = [
    "¿Cuándo fue fundada la OMS?",
    "¿Dónde tiene su sede la OMS?",
    "¿Cuál es el objetivo principal de la OMS?"
]

In [7]:
# Ejecutar QA sobre Listado
for pregunta in preguntas:
    resultado = qa_pipeline(question=pregunta, context=contexto)
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {resultado['answer']}\n")

Pregunta: ¿Cuándo fue fundada la OMS?
Respuesta: 1948

Pregunta: ¿Dónde tiene su sede la OMS?
Respuesta: Ginebra, Suiza

Pregunta: ¿Cuál es el objetivo principal de la OMS?
Respuesta: garantizar la salud global



# Fine-Tuning Modelo QA (Inglés)

In [8]:
#Importar Librerías
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from datasets import load_dataset

In [9]:
#Cargar Dataset
dataset = load_dataset("squad", split="train[:5%]")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [10]:
#Revisar Volumetría & Ejemplo
print(f"Volumetría Datos: {len(dataset)}")
print(dataset[0])

Volumetría Datos: 4380
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'answers': {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}}


In [11]:
#Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
#Función Pre-Procesamiento
def preprocess(example):
    encoding = tokenizer(
        example["question"],
        example["context"],
        truncation=True,
        padding="max_length",
        max_length=384,
        return_offsets_mapping=True
    )

    start_char = example["answers"]["answer_start"][0]
    end_char = start_char + len(example["answers"]["text"][0])
    offsets = encoding["offset_mapping"]

    start_token = end_token = 0
    for i, (start, end) in enumerate(offsets):
        if start <= start_char < end:
            start_token = i
        if start < end_char <= end:
            end_token = i
            break

    encoding["start_positions"] = start_token
    encoding["end_positions"] = end_token
    encoding.pop("offset_mapping")
    return encoding

In [13]:
#Tokenización
tokenized_dataset = dataset.map(preprocess)

Map:   0%|          | 0/4380 [00:00<?, ? examples/s]

In [14]:
#Modelo QA
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
#Configurar Entrenamiento
training_args = TrainingArguments(
    output_dir="./qa_finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [16]:
#Fine-Tuning Modelo QA
trainer.train()

Step,Training Loss
100,4.267700
200,3.530500
300,2.789100
400,2.493600
500,2.285800


TrainOutput(global_step=548, training_loss=3.003725692303511, metrics={'train_runtime': 171.64, 'train_samples_per_second': 25.519, 'train_steps_per_second': 3.193, 'total_flos': 429195433605120.0, 'train_loss': 3.003725692303511, 'epoch': 1.0})

In [17]:
#Predicción de Nuevo Ejemplo (Contexto + Pregunta)
context = """
The Great Wall of China is a series of fortifications built along the northern borders of China.
Its purpose was to protect Chinese states against invasions. It stretches over 13,000 miles.
"""

question = "What was the purpose of the Great Wall of China?"

In [18]:
#Preparar Inputs
inputs = tokenizer(question, context, return_tensors="pt", truncation=True, padding=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [19]:
#Predicción
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    start = torch.argmax(outputs.start_logits)
    end = torch.argmax(outputs.end_logits)
    answer = tokenizer.decode(inputs["input_ids"][0][start:end+1])

In [20]:
#Revisar Respuesta
print("Question:", question)
print("Answer:", answer)

Question: What was the purpose of the Great Wall of China?
Answer: invasions
